# Module 1: size reveal with Delta Analyzer

**Presenter demo (browser).** Run this in the Lab 1 debrief to put real **bytes** on two of the five planted issues in the `01 Baseline (messy)` model:

- the unused, high-cardinality free-text **`Notes`** column (planted issue 5)
- the **`ProductName`** text join key, versus an integer key (planted issue 4)

**Why not VertiPaq Analyzer?** On a Direct Lake model it reports 0 bytes for every column, because the size columns are disabled upstream in OneLake for a security reason. Cardinality and encoding still show, but the four size columns are blank. Delta Analyzer reads the Delta and Parquet files directly, so it shows the real compressed size per column, with no desktop tools.

> Run this as a **PySpark** notebook, and ideally **before the session** so the interactive tables are already rendered and you just scroll to them on stage. Allow one to two minutes on a 3M-row fact.

## 1. Install Semantic Link Labs

In [ ]:
%pip install -q semantic-link-labs

## 2. Configure

Point at the shared lakehouse that holds the generated tables. Names, not GUIDs, so this stays re-deliverable.

In [ ]:
import sempy_labs as labs

# The shared lakehouse that holds the generated tables.
SHARED_WORKSPACE = "Successful Semantic Modelling"
SHARED_LAKEHOUSE = "workshop_shared"

MESSY_TABLE = "sales_messy"   # the Module 1 diagnose fact (bad design)
CLEAN_TABLE = "sales"         # the clean order-line fact (good design)

## 3. Before: the messy fact

Open the **Columns** tab in the result and sort by **Compressed Size**. `Notes` and `ProductName` sit at the top: a column nobody uses, and a text key that will never compress like an integer. `skip_cardinality=False` adds the Cardinality column, and `Notes` is close to one distinct value per row, which is exactly why it is so expensive.

In [ ]:
_ = labs.delta_analyzer(
    MESSY_TABLE,
    lakehouse=SHARED_LAKEHOUSE, # workspace=SHARED_WORKSPACE,
    skip_cardinality=False, approx_distinct_count=True)

## 4. After: the clean fact

The same rows, modelled well: integer surrogate keys and no free-text Notes column. Compare the total size and the per-column sizes against the messy fact.

In [ ]:
_ = labs.delta_analyzer(
    CLEAN_TABLE,
    lakehouse=SHARED_LAKEHOUSE, #workspace=SHARED_WORKSPACE,
    skip_cardinality=False, approx_distinct_count=True)

## Talking points

- `Notes` is often the single biggest column in the messy fact, for zero analytical value. That is "cardinality is the hidden cost", made concrete.
- `ProductName` as a text join key costs more than an integer `ProductKey`, and is fragile to renames. That is the surrogate-key argument, in bytes.
- The messy and clean facts hold the **same rows**, so any size difference is pure **design**, not data volume.
- This is the browser-native stand-in for VertiPaq Analyzer, which reports 0 for Direct Lake columns.